# Syntext: Large-Scale Bigram Weight Table Generator

Generates `src/tokenizer/weights.rs` — the bigram frequency weight table
used by the sparse n-gram tokenizer.

**Why this notebook?** The default `scripts/weights_gen.py` trains on ~175 MB
from `the-stack-smol`. This notebook streams from
[`bigcode/the-stack-dedup`](https://huggingface.co/datasets/bigcode/the-stack-dedup)
(3 TB+, deduplicated, multilingual, permissively licensed) to produce a
higher-quality weight table trained on 10–100 GB+ of real-world source code.

**Before running — you must request dataset access:**
1. Visit https://huggingface.co/datasets/bigcode/the-stack-dedup
2. Click **"Agree and access repository"** (requires a free HuggingFace account)
3. Access is usually granted instantly
4. Then run this notebook

The notebook validates access in section 4 and will stop with clear
instructions if you haven't completed this step.

## 1. Install dependencies

In [ ]:
!pip install -q datasets huggingface_hub numpy

## 2. Authenticate with HuggingFace

**Setup (one time):**
1. Create a token at https://huggingface.co/settings/tokens (read-only is fine)
2. In Colab: click the 🔑 key icon in the left sidebar → **Add new secret**
3. Name: `HF_TOKEN`, Value: paste your token
4. Toggle the notebook access switch **ON**

If you prefer a manual prompt, set `USE_COLAB_SECRETS = False` below.

In [ ]:
from huggingface_hub import login

USE_COLAB_SECRETS = True

if USE_COLAB_SECRETS:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        login(token=HF_TOKEN)
        print("Authenticated via Colab secrets.")
    except Exception as e:
        print(f"Could not read HF_TOKEN from Colab secrets: {e}")
        print("Falling back to manual login...")
        print("(Or add HF_TOKEN to Colab secrets: key icon in left sidebar)")
        login()
else:
    login()

## 3. Configuration

| Target | Approx time (Colab free) | Quality |
|--------|--------------------------|----------|
| 1 GB   | ~15 min                  | Good — 5x the default |
| 10 GB  | ~2.5 hr                  | Very good |
| 50 GB  | ~12 hr                   | Excellent |
| 100 GB | ~24 hr                   | Near-optimal convergence |

The weight table converges well by ~10 GB. Beyond 50 GB the marginal
improvement is small.

In [ ]:
import numpy as np
import time
import os

# ── Tunables ──────────────────────────────────────────────────────────────

# Total bytes of source code to process across all languages.
TARGET_BYTES = 10 * 1_000_000_000  # 10 GB

# Primary dataset: the-stack-dedup (3 TB+, deduplicated, multilingual).
# This is gated — you must accept terms at the HuggingFace page first.
#
# If you cannot get access, set this to the ungated fallback:
#   DATASET_NAME = "codeparrot/github-code-clean"
# Note: codeparrot is smaller and Python-focused, so the resulting
# weight table will be lower quality for non-Python languages.
DATASET_NAME = "bigcode/the-stack-dedup"

# Languages and their sampling weights (higher = more bytes).
# Tier 1: agent-heavy languages get a larger share.
# Tier 2: additional languages for broader bigram coverage.
LANGUAGES = {
    # Tier 1
    "python":     2.0,
    "javascript": 2.0,
    "typescript": 2.0,
    "rust":       2.0,
    "go":         1.5,
    "java":       1.5,
    "c":          1.5,
    "c++":        1.5,
    # Tier 2
    "c-sharp":    1.0,
    "ruby":       1.0,
    "php":        1.0,
    "swift":      0.8,
    "kotlin":     0.8,
    "scala":      0.7,
    "shell":      0.7,
    "lua":        0.5,
    "haskell":    0.5,
    "r":          0.5,
    "perl":       0.4,
    "dart":       0.4,
    "elixir":     0.3,
    "julia":      0.3,
    "ocaml":      0.3,
}

# Max bytes per file (skip minified JS, generated code, etc.).
MAX_FILE_BYTES = 500_000

# Checkpoint interval: save progress every N bytes.
# Protects against Colab disconnections.
CHECKPOINT_INTERVAL = 500_000_000  # 500 MB

CHECKPOINT_PATH = "/content/bigram_checkpoint.npz"
OUTPUT_PATH = "/content/weights.rs"

# ── Dataset-specific config ────────────────────────────────────────────────
# Maps dataset name -> how to load it and which field has the source code.
#
# the-stack-dedup: each language is a separate data_dir shard.
# codeparrot:      uses a `languages` filter parameter, content in "code".

DATASET_CONFIGS = {
    "bigcode/the-stack-dedup": {
        "content_field": "content",
        "load_fn": lambda ds_name, lang: (
            __import__('datasets').load_dataset(
                ds_name,
                data_dir=f"data/{lang}",
                split="train",
                streaming=True,
            )
        ),
        # the-stack-dedup uses lowercase language names as directory shards.
        "lang_names": {k: k for k in LANGUAGES},
    },
    "codeparrot/github-code-clean": {
        "content_field": "code",
        "load_fn": lambda ds_name, lang: (
            __import__('datasets').load_dataset(
                ds_name,
                split="train",
                streaming=True,
                languages=[lang],
            )
        ),
        "lang_names": {
            "python": "Python", "javascript": "JavaScript",
            "typescript": "TypeScript", "rust": "Rust",
            "go": "Go", "java": "Java", "c": "C", "c++": "C++",
            "c-sharp": "C#", "ruby": "Ruby", "php": "PHP",
            "swift": "Swift", "kotlin": "Kotlin", "scala": "Scala",
            "shell": "Shell", "lua": "Lua", "haskell": "Haskell",
            "r": "R", "perl": "Perl", "dart": "Dart",
            "elixir": "Elixir", "julia": "Julia", "ocaml": "OCaml",
        },
    },
}

assert DATASET_NAME in DATASET_CONFIGS, (
    f"Unknown dataset '{DATASET_NAME}'. "
    f"Known: {', '.join(DATASET_CONFIGS)}"
)
ds_config = DATASET_CONFIGS[DATASET_NAME]

print(f"Target:  {TARGET_BYTES / 1e9:.1f} GB across {len(LANGUAGES)} languages")
print(f"Dataset: {DATASET_NAME}")
print(f"Checkpoint every {CHECKPOINT_INTERVAL / 1e6:.0f} MB")

## 4. Validate dataset access

Probes the dataset before starting the hours-long streaming loop.
If access is denied, stops immediately with instructions.

In [ ]:
def validate_access(dataset_name: str, ds_config: dict, languages: dict) -> list:
    """Probe the dataset. Returns list of accessible language keys.

    Raises RuntimeError with actionable message if the dataset is gated
    and the user hasn't been granted access yet.
    """
    lang_names = ds_config["lang_names"]
    load_fn = ds_config["load_fn"]
    content_field = ds_config["content_field"]

    # ── Phase 1: auth + gating check on one language ─────────────────────
    first_lang = next(iter(languages))
    mapped = lang_names.get(first_lang, first_lang)
    print(f"Probing {dataset_name} (language='{mapped}')...")

    try:
        ds = load_fn(dataset_name, mapped)
        sample = next(iter(ds))
    except Exception as e:
        err = str(e).lower()
        if "gated" in err or "access" in err or "authorization" in err:
            print()
            print("=" * 64)
            print(f"  ACCESS DENIED: {dataset_name}")
            print("=" * 64)
            print()
            print("  This dataset requires you to accept its terms of use.")
            print()
            print("  Steps to fix:")
            print(f"    1. Open https://huggingface.co/datasets/{dataset_name}")
            print("    2. Click 'Agree and access repository'")
            print("    3. Access is usually granted instantly")
            print("    4. Re-run this cell")
            print()
            print("  Make sure your HF_TOKEN (section 2) belongs to the")
            print("  same account that accepted the terms.")
            print()
            print("  If you cannot get access, switch to the ungated fallback")
            print("  in section 3 (lower quality, Python-focused):")
            print('    DATASET_NAME = "codeparrot/github-code-clean"')
            print("=" * 64)
            raise RuntimeError(
                f"Cannot access {dataset_name}. "
                f"Accept terms at https://huggingface.co/datasets/{dataset_name}"
            ) from e
        else:
            raise

    # Verify the content field exists.
    if content_field not in sample:
        available = list(sample.keys())
        raise RuntimeError(
            f"Field '{content_field}' not in sample. "
            f"Available: {available}. "
            f"Update DATASET_CONFIGS['{dataset_name}']['content_field']."
        )

    content_len = len(sample[content_field])
    print(f"  Access confirmed ({content_len} chars in first sample).")

    # ── Phase 2: probe each language ─────────────────────────────────────
    print(f"Checking {len(languages)} languages...")
    accessible = []
    skipped = []

    for lang in languages:
        mapped = lang_names.get(lang, lang)
        try:
            ds = load_fn(dataset_name, mapped)
            next(iter(ds))
            accessible.append(lang)
        except StopIteration:
            skipped.append((lang, "empty — no data for this language"))
        except Exception as e:
            skipped.append((lang, str(e)[:100]))

    print(f"\n  Accessible: {len(accessible)} / {len(languages)}")
    if skipped:
        print(f"  Skipped:")
        for lang, reason in skipped:
            print(f"    {lang}: {reason}")

    if not accessible:
        raise RuntimeError(
            f"No languages accessible in {dataset_name}."
        )

    return accessible


# ── Run the check ──────────────────────────────────────────────────────────
accessible_langs = validate_access(DATASET_NAME, ds_config, LANGUAGES)
LANGUAGES = {k: v for k, v in LANGUAGES.items() if k in accessible_langs}
print(f"\nReady to stream {len(LANGUAGES)} languages: {', '.join(LANGUAGES)}")

## 5. Streaming bigram counter

Streams data without downloading the full dataset.
Memory stays constant (~600 MB for counts + streaming buffer).
Checkpoints to disk for crash recovery.

In [ ]:
def count_bigrams_streaming(
    dataset_name,
    ds_config,
    languages,
    target_bytes,
    max_file_bytes,
    checkpoint_path,
    checkpoint_interval,
):
    """Stream source code and count bigram frequencies.

    Returns (counts_array[65536], total_bytes_processed).
    """
    content_field = ds_config["content_field"]
    lang_names = ds_config["lang_names"]
    load_fn = ds_config["load_fn"]

    # Resume from checkpoint if one exists.
    if os.path.exists(checkpoint_path):
        ckpt = np.load(checkpoint_path, allow_pickle=False)
        counts = ckpt["counts"]
        total_bytes = int(ckpt["total_bytes"])
        lang_bytes_done = dict(zip(
            [str(n) for n in ckpt["lang_names"]],
            [int(b) for b in ckpt["lang_bytes"]],
        ))
        print(f"Resumed from checkpoint: {total_bytes / 1e9:.2f} GB already processed")
    else:
        counts = np.zeros(65536, dtype=np.int64)
        total_bytes = 0
        lang_bytes_done = {}

    # Per-language byte targets proportional to weights.
    total_weight = sum(languages.values())
    lang_targets = {
        lang: int(target_bytes * weight / total_weight)
        for lang, weight in languages.items()
    }

    last_checkpoint = total_bytes
    wall_start = time.time()

    for lang, byte_target in lang_targets.items():
        already = lang_bytes_done.get(lang, 0)
        if already >= byte_target:
            print(f"  {lang}: already done ({already / 1e6:.0f} MB)")
            continue

        lang_bytes = already
        remaining = byte_target - already
        mapped = lang_names.get(lang, lang)
        print(f"  {lang}: streaming {remaining / 1e6:.0f} MB "
              f"(target {byte_target / 1e6:.0f} MB) ...")

        try:
            ds = load_fn(dataset_name, mapped)
        except Exception as e:
            print(f"    SKIP {lang}: {e}")
            continue

        file_count = 0
        skipped_large = 0

        for sample in ds:
            if lang_bytes >= byte_target:
                break

            content = sample.get(content_field, "")
            if not content:
                continue

            raw = content.encode("utf-8", errors="replace")
            if len(raw) > max_file_bytes:
                skipped_large += 1
                continue

            # Lowercase ASCII bytes, leave non-ASCII as-is (matches syntext).
            lowered = bytes(
                b | 0x20 if 0x41 <= b <= 0x5A else b for b in raw
            )

            n = len(lowered)
            if n < 2:
                continue

            # Vectorized bigram counting.
            arr = np.frombuffer(lowered, dtype=np.uint8)
            pairs = arr[:-1].astype(np.uint16) << 8 | arr[1:].astype(np.uint16)
            np.add.at(counts, pairs, 1)

            lang_bytes += n
            total_bytes += n
            file_count += 1

            if total_bytes - last_checkpoint >= checkpoint_interval:
                _save_checkpoint(
                    checkpoint_path, counts, total_bytes,
                    lang_bytes_done, lang, lang_bytes,
                )
                last_checkpoint = total_bytes
                elapsed = time.time() - wall_start
                rate = total_bytes / elapsed / 1e6 if elapsed > 0 else 0
                pct = total_bytes / target_bytes * 100
                eta_min = (target_bytes - total_bytes) / (total_bytes / elapsed) / 60 if total_bytes > 0 else 0
                print(
                    f"    checkpoint: {total_bytes / 1e9:.2f} GB "
                    f"({pct:.1f}%) | {rate:.1f} MB/s | "
                    f"ETA {eta_min:.0f} min | {lang}"
                )

        lang_bytes_done[lang] = lang_bytes
        if skipped_large:
            print(f"    ({skipped_large} files > {max_file_bytes // 1000} KB skipped)")
        print(f"    done: {lang_bytes / 1e6:.0f} MB, {file_count} files")

    # Final checkpoint.
    _save_checkpoint(
        checkpoint_path, counts, total_bytes, lang_bytes_done, "", 0
    )

    elapsed = time.time() - wall_start
    print(f"\nFinished: {total_bytes / 1e9:.2f} GB in {elapsed / 60:.1f} min")
    if elapsed > 0:
        print(f"Throughput: {total_bytes / elapsed / 1e6:.1f} MB/s")

    return counts, total_bytes


def _save_checkpoint(path, counts, total_bytes, lang_bytes_done, current_lang, current_bytes):
    merged = dict(lang_bytes_done)
    if current_lang:
        merged[current_lang] = current_bytes
    np.savez_compressed(
        path,
        counts=counts,
        total_bytes=np.array(total_bytes),
        lang_names=np.array(list(merged.keys())),
        lang_bytes=np.array(list(merged.values())),
    )

## 6. Run the bigram counter

Checkpoints periodically. If Colab disconnects, **re-run this cell** to
resume from the last checkpoint.

To restart from scratch: `!rm /content/bigram_checkpoint.npz`

In [ ]:
counts, total_bytes = count_bigrams_streaming(
    dataset_name=DATASET_NAME,
    ds_config=ds_config,
    languages=LANGUAGES,
    target_bytes=TARGET_BYTES,
    max_file_bytes=MAX_FILE_BYTES,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_interval=CHECKPOINT_INTERVAL,
)

print(f"\nNon-zero bigram pairs: {np.count_nonzero(counts):,}")
print(f"Total pair occurrences: {counts.sum():,}")

## 7. Convert counts to weights

In [ ]:
def counts_to_weights(counts):
    """Invert frequencies: rare pairs -> high weight, common -> low."""
    nonzero_mask = counts > 0
    if not nonzero_mask.any():
        return np.full(65536, 65535, dtype=np.uint16)

    log_counts = np.log1p(counts.astype(np.float64))
    max_log = log_counts.max()
    if max_log == 0:
        return np.full(65536, 65535, dtype=np.uint16)

    normalized = log_counts / max_log
    weights = ((1.0 - normalized) * 64900 + 100).astype(np.float64)
    weights[~nonzero_mask] = 65535

    return np.clip(weights, 0, 65535).astype(np.uint16)


weights = counts_to_weights(counts)
print("Weight conversion complete.")

## 8. Diagnostics

In [ ]:
def print_diagnostics(counts, weights, total_bytes):
    print("=" * 60)
    print(f"DIAGNOSTICS  ({total_bytes / 1e9:.1f} GB corpus)")
    print("=" * 60)

    if counts.sum() == 0:
        print("\nALL COUNTS ARE ZERO — no data was processed.")
        print("Go back to section 4 and check dataset access.")
        return

    top = np.argsort(counts)[::-1][:20]
    print("\nTop 20 most common byte pairs:")
    for idx in top:
        b1, b2 = idx >> 8, idx & 0xFF
        c1 = chr(b1) if 32 <= b1 < 127 else f"\\x{b1:02x}"
        c2 = chr(b2) if 32 <= b2 < 127 else f"\\x{b2:02x}"
        print(f"  '{c1}{c2}'  count={counts[idx]:>14,}  weight={weights[idx]:>5}")

    nonzero = counts > 0
    rare = np.argsort(counts + (~nonzero).astype(np.int64) * 10**18)[:20]
    print("\nTop 20 rarest (nonzero) byte pairs:")
    for idx in rare:
        if counts[idx] == 0:
            continue
        b1, b2 = idx >> 8, idx & 0xFF
        c1 = chr(b1) if 32 <= b1 < 127 else f"\\x{b1:02x}"
        c2 = chr(b2) if 32 <= b2 < 127 else f"\\x{b2:02x}"
        print(f"  '{c1}{c2}'  count={counts[idx]:>8,}  weight={weights[idx]:>5}")

    print(f"\nDistribution:")
    print(f"  Non-zero pairs:       {nonzero.sum():,}")
    print(f"  Zero-count pairs:     {(~nonzero).sum():,}")
    print(f"  Weight min (nonzero): {weights[nonzero].min()}")
    print(f"  Weight max (nonzero): {weights[nonzero].max()}")
    print(f"  Weight median:        {int(np.median(weights))}")

    print("\nSanity check:")
    checks = [
        (ord(' '), ord(' '), '"  "  double space'),
        (ord('e'), ord(' '), '"e "  common word end'),
        (ord('r'), ord('e'), '"re"  return, result'),
        (ord('i'), ord('n'), '"in"  in, int, include'),
        (ord('f'), ord('n'), '"fn"  Rust keyword'),
        (ord('_'), ord('_'), '"__"  Python dunder'),
        (ord('q'), ord('z'), '"qz"  rare'),
        (ord('x'), ord('j'), '"xj"  rare'),
    ]
    for b1, b2, label in checks:
        idx = (b1 << 8) | b2
        print(f"  {label:25s}  count={counts[idx]:>14,}  weight={weights[idx]:>5}")

    # Pass/fail gates matching scripts/weights_gen.py expectations.
    common_ok = all(
        weights[(b1 << 8) | b2] < 12000
        for b1, b2 in [(ord(' '), ord(' ')), (ord('r'), ord('e')), (ord('e'), ord('r'))]
    )
    rare_ok = all(
        weights[(b1 << 8) | b2] > 35000
        for b1, b2 in [(ord('q'), ord('z')), (ord('x'), ord('j'))]
    )
    nonzero_ok = nonzero.sum() > 15000

    print(f"\n  Common pairs < 12000:  {'PASS' if common_ok else 'FAIL'}")
    print(f"  Rare pairs > 35000:    {'PASS' if rare_ok else 'FAIL'}")
    print(f"  Non-zero pairs > 15k:  {'PASS' if nonzero_ok else 'FAIL'} ({nonzero.sum():,})")

    if common_ok and rare_ok and nonzero_ok:
        print("\n  All checks passed.")
    else:
        print("\n  Some checks failed — consider processing more data.")


print_diagnostics(counts, weights, total_bytes)

## 9. Emit `weights.rs`

In [ ]:
def emit_rust_const(weights, total_bytes, dataset_name, output_path):
    if (weights == 65535).all():
        print("Cannot write weights.rs — no data was processed.")
        return

    with open(output_path, "w") as f:
        f.write("// Auto-generated by weights_gen_colab.ipynb\n")
        f.write("// Bigram frequency weights for sparse n-gram tokenization.\n")
        f.write("// Rare byte-pairs get high weights (good gram boundaries).\n")
        f.write("// Common byte-pairs get low weights (gram interiors).\n")
        f.write("//\n")
        f.write("// Index: (byte_a << 8) | byte_b\n")
        f.write("// Usage: BIGRAM_WEIGHTS[(b1 as usize) << 8 | (b2 as usize)]\n")
        f.write(f"// Corpus: {total_bytes / 1e9:.1f} GB from {dataset_name}\n")
        f.write("//         (Rust, Python, JS, TS, Go, Java, C, C++, and more)\n")
        f.write("//\n")
        f.write("// DO NOT EDIT BY HAND.\n")
        f.write("\n")
        f.write("/// Pre-trained byte-pair frequency table.\n")
        f.write("pub const BIGRAM_WEIGHTS: [u16; 65536] = [\n")
        for i in range(0, 65536, 16):
            row = ", ".join(f"{weights[i+j]:5}" for j in range(16))
            f.write(f"    {row},\n")
        f.write("];\n")

    size_kb = os.path.getsize(output_path) / 1024
    print(f"Wrote {output_path} ({size_kb:.0f} KB)")


emit_rust_const(weights, total_bytes, DATASET_NAME, OUTPUT_PATH)

## 10. Download

In [ ]:
try:
    from google.colab import files
    if os.path.exists(OUTPUT_PATH):
        files.download(OUTPUT_PATH)
        print("Download started. Check your browser.")
    else:
        print("No weights.rs found — run sections 6-9 first.")
except ImportError:
    print(f"Not in Colab. File at: {OUTPUT_PATH}")

## 11. (Optional) Compare against current weights

Upload your existing `weights.rs` to `/content/weights_old.rs`, then
uncomment the last two lines.

In [ ]:
import re as _re

def parse_existing_weights(path):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        text = f.read()
    match = _re.search(r'\[([^\]]+)\]', text, _re.DOTALL)
    if not match:
        return None
    nums = [int(x.strip()) for x in match.group(1).split(',') if x.strip()]
    if len(nums) != 65536:
        print(f"Warning: expected 65536 entries, got {len(nums)}")
        return None
    return np.array(nums, dtype=np.uint16)

def compare_weights(old, new):
    diff = new.astype(np.int32) - old.astype(np.int32)
    changed = np.count_nonzero(diff)
    print(f"Changed: {changed:,} / 65,536 ({changed / 655.36:.1f}%)")
    print(f"Max increase: {diff.max():+d}  Max decrease: {diff.min():+d}")
    print(f"Mean |diff|: {np.abs(diff).mean():.1f}")
    top = np.argsort(np.abs(diff))[::-1][:15]
    print("\nBiggest changes:")
    for idx in top:
        b1, b2 = idx >> 8, idx & 0xFF
        if 32 <= b1 < 127 and 32 <= b2 < 127:
            print(f"  '{chr(b1)}{chr(b2)}'  {old[idx]:5} -> {new[idx]:5}  ({diff[idx]:+d})")

# old_weights = parse_existing_weights("/content/weights_old.rs")
# if old_weights is not None: compare_weights(old_weights, weights)
print("Upload weights_old.rs and uncomment the lines above.")

## 12. (Optional) Save raw counts

In [ ]:
np.savez_compressed("/content/bigram_counts_final.npz",
                    counts=counts, total_bytes=np.array(total_bytes))
print(f"Saved ({os.path.getsize('/content/bigram_counts_final.npz') / 1e6:.1f} MB)")

try:
    from google.colab import files
    files.download("/content/bigram_counts_final.npz")
except ImportError:
    pass

---

## After generating

```bash
cp ~/Downloads/weights.rs src/tokenizer/weights.rs
cargo test
cargo bench --bench query_latency -- --sample-size 10
cargo bench --bench index_build -- --sample-size 10
python3 scripts/bench_compare.py --preset react_token_aligned
```

Record before/after in `docs/BENCHMARKS.md`.